<a href="https://colab.research.google.com/github/prathameshmowade/Patern-Recognition-/blob/main/PR3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# SMS Spam Detection using Naive Bayes
# ==========================================

# Install required package
!pip install ucimlrepo -q

# Import libraries
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ------------------------------------------
# Load Dataset
# ------------------------------------------

# Download and unzip the dataset
!wget -nc https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
!unzip -n smsspamcollection.zip

# Load the dataset into a DataFrame
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

print("First 5 Records of the combined DataFrame:")
print(df.head())

print("\nTarget (Label) column:")
print(df["label"].head())

print("\nDataset Shape:", df.shape)

print("\nClass Distribution")
print(df["label"].value_counts())

# ------------------------------------------
# Convert Labels
# ham = 0
# spam = 1
# ------------------------------------------

df["label"] = df["label"].map({
    "ham":0,
    "spam":1
})

# ------------------------------------------
# Split Dataset
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42
)

# ------------------------------------------
# Feature Extraction
# ------------------------------------------

vectorizer = CountVectorizer(stop_words='english')

X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

# ------------------------------------------
# Train Naive Bayes Model
# ------------------------------------------

model = MultinomialNB()

model.fit(X_train_vector, y_train)

# ------------------------------------------
# Prediction
# ------------------------------------------

y_pred = model.predict(X_test_vector)

# ------------------------------------------
# Accuracy
# ------------------------------------------

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

# ------------------------------------------
# Classification Report
# ------------------------------------------

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# ------------------------------------------
# Confusion Matrix
# ------------------------------------------

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# ------------------------------------------
# Test Custom SMS
# ------------------------------------------

sample = [
    "Congratulations! You have won a FREE lottery prize. Call now.",
    "Are you coming to college today?",
    "Claim your reward by clicking the link.",
    "Let's meet for lunch tomorrow."
]

sample_vector = vectorizer.transform(sample)

prediction = model.predict(sample_vector)

print("\nCustom Predictions")

for msg, pred in zip(sample, prediction):
    print("----------------------------------")
    print("Message :", msg)
    print("Prediction :", "Spam" if pred==1 else "Ham")

# ------------------------------------------
# Most Important Words
# ------------------------------------------

feature_names = np.array(vectorizer.get_feature_names_out())

spam_prob = model.feature_log_prob_[1]
ham_prob = model.feature_log_prob_[0]

spam_words = feature_names[np.argsort(spam_prob)[-20:]]
ham_words = feature_names[np.argsort(ham_prob)[-20:]]

print("\nTop 20 Spam Words")
print(spam_words)

print("\nTop 20 Ham Words")
print(ham_words)

In [ ]:
# ==========================================
# VISUALIZATIONS
# ==========================================

import matplotlib.pyplot as plt
import numpy as np # Ensure numpy is imported for argsort
from sklearn.metrics import confusion_matrix # Ensure confusion_matrix is available

# ------------------------------------------
# 1. Class Distribution
# ------------------------------------------

class_counts = df["label"].value_counts()

plt.figure(figsize=(5,5))
plt.bar(["Ham","Spam"], class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Email Type")
plt.ylabel("Number of Messages")

for i, v in enumerate(class_counts.values):
    plt.text(i, v+20, str(v), ha='center')

plt.show()


# ------------------------------------------
# 2. Confusion Matrix
# ------------------------------------------

# Calculate confusion matrix (cm was undefined)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

plt.title("Confusion Matrix")
plt.colorbar()

plt.xticks([0,1],["Ham","Spam"])
plt.yticks([0,1],["Ham","Spam"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j],
                 ha="center",
                 va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", # Improve text visibility
                 fontsize=12)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ------------------------------------------
# 3. Top 10 Spam Words
# ------------------------------------------

# Get indices of top spam words (spam_index, spam were undefined)
spam_indices = np.argsort(spam_prob)[-10:]
spam_top_words = feature_names[spam_indices]
spam_top_values = spam_prob[spam_indices]

plt.figure(figsize=(10,5))
plt.bar(spam_top_words, spam_top_values)

plt.title("Top 10 Spam Words (Log Probability)") # Clarify y-axis meaning
plt.xlabel("Words")
plt.ylabel("Log Probability")

plt.xticks(rotation=45, ha='right') # Rotate for better readability
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()


# ------------------------------------------
# 4. Top 10 Ham Words
# ------------------------------------------

# Get indices of top ham words (ham_index, ham were undefined)
ham_indices = np.argsort(ham_prob)[-10:]
ham_top_words = feature_names[ham_indices]
ham_top_values = ham_prob[ham_indices]

plt.figure(figsize=(10,5))
plt.bar(ham_top_words, ham_top_values)

plt.title("Top 10 Ham Words (Log Probability)") # Clarify y-axis meaning
plt.xlabel("Words")
plt.ylabel("Log Probability")

plt.xticks(rotation=45, ha='right') # Rotate for better readability
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()


# ------------------------------------------
# 5. Accuracy Pie Chart
# ------------------------------------------

correct = (y_test == y_pred).sum()
incorrect = len(y_test) - correct

plt.figure(figsize=(5,5))

plt.pie(
    [correct, incorrect],
    labels=["Correct","Incorrect"],
    autopct="%1.1f%%",
    startangle=90,
    colors=['lightgreen', 'lightcoral'] # Add colors for better distinction
)

plt.title("Prediction Accuracy")

plt.show()